# Illumicell AI — Blood Cell Classifier (Option A)

**2026 Venture & Tech Summer Program · Technical Track**

This notebook teaches a computer-vision model to look at a microscope image of a white blood cell and identify its type. It uses the public **Kaggle Blood Cell Images** dataset (~12,500 labeled images, 4 cell types). No real patient data is used.

It is organized in the same order your weekly guidelines ask for:

1. **Setup** — turn on the GPU, import libraries (Week 2, Day 1)
2. **Load** — read the full dataset, confirm categories and counts (Week 2, Day 1)
3. **Explore** — show labeled examples, check class balance (Week 2, Day 2)
4. **Prepare** — resize + normalize images, encode labels, save the mapping (Week 2, Day 3)
5. **Split** — a fair, reproducible train/test split (Week 2, Day 4)
6. **Train** — transfer learning on a pre-trained model (Week 3, Day 1)
7. **Evaluate** — accuracy + confusion matrix (Week 3, Days 2-3)
8. **Study failures** — look at wrong predictions (Week 3, Day 4)
9. **Improve** — data augmentation, re-measure (Week 4, Day 1)
10. **Save** — export the model + label map for the Streamlit app (Week 4)

> **How to run:** `Runtime → Change runtime type → GPU`, then `Runtime → Run all`. Do the one-time dataset step in Section 1 first.

## 1. Setup — GPU + libraries

Enable the GPU first: **Runtime → Change runtime type → Hardware accelerator → GPU**. Then run this cell. It should print a GPU device; if it prints an empty list, the GPU is not on and training will be very slow.

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import json, os, pathlib

print('TensorFlow version:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('GPUs visible to TensorFlow:', gpus)
if not gpus:
    print('WARNING: No GPU detected. Go to Runtime > Change runtime type > GPU, then Run all again.')
else:
    print('GPU is ready. Training will be fast.')

### 1a. Get the dataset (one-time step)

The Kaggle dataset is `paultimothymooney/blood-cells`. Pick **one** of the two options below.

**Option 1 — Kaggle API (recommended).** In Kaggle: *Account → Settings → Create New Token*. This downloads `kaggle.json`. Upload it when the cell asks, and the dataset downloads automatically.

**Option 2 — manual upload.** Download the dataset zip from the Kaggle page yourself, then use the Colab file panel (folder icon on the left) to upload it, and unzip it with `!unzip yourfile.zip`.

Either way, you want a folder that contains `dataset2-master/images/TRAIN` with four subfolders (EOSINOPHIL, LYMPHOCYTE, MONOCYTE, NEUTROPHIL).

In [ ]:
# --- Option 1: Kaggle API ---
# Run this cell, click 'Choose Files', and select your kaggle.json.
from google.colab import files
import os

if not os.path.exists('/root/.kaggle/kaggle.json'):
    print('Upload your kaggle.json (from Kaggle > Account > Create New Token):')
    uploaded = files.upload()  # choose kaggle.json
    os.makedirs('/root/.kaggle', exist_ok=True)
    !cp kaggle.json /root/.kaggle/kaggle.json
    !chmod 600 /root/.kaggle/kaggle.json

!pip -q install kaggle
!kaggle datasets download -d paultimothymooney/blood-cells -p /content --unzip
print('Done. Contents of /content:')
!ls /content

In [ ]:
# Point this at the TRAIN folder. Adjust if your unzip put it somewhere else.
DATA_DIR = '/content/dataset2-master/dataset2-master/images/TRAIN'
if not os.path.isdir(DATA_DIR):
    # common alternate layout
    alt = '/content/dataset2-master/images/TRAIN'
    DATA_DIR = alt if os.path.isdir(alt) else DATA_DIR
print('Using DATA_DIR =', DATA_DIR)
assert os.path.isdir(DATA_DIR), 'TRAIN folder not found. Check where the zip unpacked (see the ls output above).'
print('Classes found:', sorted(os.listdir(DATA_DIR)))

## 2-3. Load & explore the data

We load images straight from the folders with `image_dataset_from_directory`, which uses each subfolder name as the label. We hold out **20%** as a test set the model never trains on, with a **fixed seed** so the split is identical every run (Week 2, Day 4).

`IMG_SIZE` standardizes every image to the same shape; the model needs consistent input.

In [ ]:
IMG_SIZE = (128, 128)   # every image resized to this
BATCH_SIZE = 32
SEED = 42               # fixed seed => reproducible split

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR, validation_split=0.2, subset='training', seed=SEED,
    image_size=IMG_SIZE, batch_size=BATCH_SIZE)

test_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR, validation_split=0.2, subset='validation', seed=SEED,
    image_size=IMG_SIZE, batch_size=BATCH_SIZE)

class_names = train_ds.class_names
num_classes = len(class_names)
print('Categories (label order):', class_names)
print('Number of classes:', num_classes)

### Class balance
A very uneven dataset is harder to train on and can make accuracy misleading. Count how many images are in each category (Week 2, Day 1).

In [ ]:
from collections import Counter
counts = Counter()
for cls in class_names:
    counts[cls] = len(os.listdir(os.path.join(DATA_DIR, cls)))
for cls, n in counts.items():
    print(f'{cls:12s}: {n} images')

plt.figure(figsize=(6,3))
plt.bar(list(counts.keys()), list(counts.values()))
plt.title('Images per category'); plt.ylabel('count'); plt.xticks(rotation=20)
plt.tight_layout(); plt.show()

### Look at real examples
Seeing the images with your own eyes prevents silent labeling mistakes. Note which two categories look most alike — those are the ones the model will most likely confuse (Week 2, Day 2).

In [ ]:
plt.figure(figsize=(10,10))
for images, labels in train_ds.take(1):
    for i in range(9):
        ax = plt.subplot(3,3,i+1)
        plt.imshow(images[i].numpy().astype('uint8'))
        plt.title(class_names[labels[i]])
        plt.axis('off')
plt.tight_layout(); plt.show()

## 4. Prepare — normalize + save the label mapping

Normalizing scales pixel values into the range the pre-trained model expects. We also **save the label mapping** (number → cell-type name) to a file — treat this as a real deliverable, because without it your predictions are just unreadable numbers (Week 2, Day 3).

In [ ]:
# Save the number->name mapping so predictions are readable in Week 3/4 and in the app.
label_map = {i: name for i, name in enumerate(class_names)}
with open('label_map.json', 'w') as f:
    json.dump(label_map, f, indent=2)
print('Saved label_map.json:', label_map)

# Performance: cache + prefetch so the GPU is not waiting on disk.
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000, seed=SEED).prefetch(AUTOTUNE)
test_ds  = test_ds.cache().prefetch(AUTOTUNE)

## 5-6. Build & train the model (transfer learning)

We start from **MobileNetV2**, already trained on millions of images, and adapt it to our 4 cell types. Transfer learning gives far better results than training from scratch for a project this size (Week 3, Day 1).

The `data_augmentation` layer (random flips/rotations) is the Week 4 improvement built in from the start — it expands the training set and helps the model generalize. Preprocessing + normalization happen inside the model, so the Streamlit app can feed it a raw image later.

In [ ]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
], name='augment')

base_model = tf.keras.applications.MobileNetV2(
    input_shape=IMG_SIZE + (3,), include_top=False, weights='imagenet')
base_model.trainable = False  # freeze the pre-trained layers first

inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
x = data_augmentation(inputs)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)  # scales pixels to [-1,1]
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.2)(x)
outputs = tf.keras.layers.Dense(num_classes, activation='softmax')(x)
model = tf.keras.Model(inputs, outputs)

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])
model.summary()

### Train
Start with a short run to confirm the whole pipeline works end to end, then increase `EPOCHS` for a better model. Watch that the loss goes down — if it doesn't, something is wrong (Week 3, Day 1).

In [ ]:
EPOCHS = 8   # start small (e.g. 3) to confirm it runs, then raise to 8-15
history = model.fit(train_ds, validation_data=test_ds, epochs=EPOCHS)

In [ ]:
# Plot the learning curves so you can see it actually learned.
acc = history.history['accuracy']; val_acc = history.history['val_accuracy']
loss = history.history['loss']; val_loss = history.history['val_loss']
plt.figure(figsize=(10,4))
plt.subplot(1,2,1); plt.plot(acc,label='train'); plt.plot(val_acc,label='test'); plt.title('Accuracy'); plt.legend()
plt.subplot(1,2,2); plt.plot(loss,label='train'); plt.plot(val_loss,label='test'); plt.title('Loss'); plt.legend()
plt.show()

## 7. Evaluate — accuracy + confusion matrix

Now we measure on the held-out test set. **Sanity check:** with 4 equal categories, random guessing is ~25%, so a real model should be well above that. The confusion matrix shows *which* categories the model mixes up — that's where all your improvement ideas come from (Week 3, Days 2-3).

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

y_true, y_pred = [], []
for images, labels in test_ds:
    probs = model.predict(images, verbose=0)
    y_pred.extend(np.argmax(probs, axis=1))
    y_true.extend(labels.numpy())
y_true = np.array(y_true); y_pred = np.array(y_pred)

accuracy = (y_true == y_pred).mean()
print(f'TEST ACCURACY: {accuracy*100:.1f}%   (random guessing = {100/num_classes:.0f}%)')
print()
print(classification_report(y_true, y_pred, target_names=class_names))

In [ ]:
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6,5))
plt.imshow(cm, cmap='Blues')
plt.title('Confusion matrix'); plt.colorbar()
plt.xticks(range(num_classes), class_names, rotation=45, ha='right')
plt.yticks(range(num_classes), class_names)
plt.xlabel('Predicted'); plt.ylabel('True')
for i in range(num_classes):
    for j in range(num_classes):
        plt.text(j, i, cm[i,j], ha='center', va='center',
                 color='white' if cm[i,j] > cm.max()/2 else 'black')
plt.tight_layout(); plt.show()

# The two categories the model confuses most (the biggest off-diagonal cell):
cm_off = cm.copy(); np.fill_diagonal(cm_off, 0)
i, j = np.unravel_index(np.argmax(cm_off), cm_off.shape)
print(f'Most-confused pair: TRUE {class_names[i]} predicted as {class_names[j]} ({cm_off[i,j]} times)')

## 8. Study the failures
Look at images the model got wrong. Are they blurry, rare, or genuinely ambiguous? Write down the patterns — this is the evidence behind your improvement hypothesis (Week 3, Day 4).

In [ ]:
shown = 0
plt.figure(figsize=(12,6))
for images, labels in test_ds.take(4):
    probs = model.predict(images, verbose=0)
    preds = np.argmax(probs, axis=1)
    for k in range(len(images)):
        if preds[k] != labels[k].numpy() and shown < 8:
            ax = plt.subplot(2,4,shown+1)
            plt.imshow(images[k].numpy().astype('uint8'))
            plt.title(f'true {class_names[labels[k]]}\npred {class_names[preds[k]]}', fontsize=9)
            plt.axis('off')
            shown += 1
plt.suptitle('Examples the model got wrong'); plt.tight_layout(); plt.show()

## 9. Save the model + label map (for the Streamlit app)

This exports everything the Week 4 interface needs. Download both files from the Colab file panel, or save them to your shared Drive folder.

In [ ]:
model.save('blood_cell_model.keras')
print('Saved blood_cell_model.keras and label_map.json')

# Optional: copy to your shared Google Drive folder so the team has them.
# from google.colab import drive
# drive.mount('/content/drive')
# !cp blood_cell_model.keras label_map.json '/content/drive/MyDrive/YOUR_TEAM_FOLDER/'

---
### What you now have (deliverables)
- A clean train/test split with a fixed seed (honest measurement)
- A trained model with a real **test accuracy** number
- A **confusion matrix** showing which cell types get confused
- Example failures to study
- A saved model + label map for the interface

### Your improvement hypothesis (fill in for Week 3 → Week 4)
> The model most confuses **____** with **____**. I think **____** (more epochs / more data for the rare class / stronger augmentation) will help most, because **____**. I expect accuracy to go from **__%** to about **__%**.

Change `EPOCHS`, or unfreeze the top of `base_model` for fine-tuning, re-run, and record before/after.